# Understanding the BIT puzzle as GATE(f, g, h) — generators + an honest search write-up

**Open Contribution Award — Best Data / Synthetic Data Method.** Team: 147 / 4,354 (top 3.4%).

We set out to teach an LLM to **search** these reasoning puzzles — enumerate
candidate rules, test them against the examples, prune, lock. We didn't fully get
there. This notebook is **what we learned along the way** and the **base generators**
we built to explore it, shared as honest work-in-progress.

Every generated row has a known latent rule and an exact gold answer (we sampled the
rule), so in our working system each forward generator was paired with a solver/verifier
that acted as a free oracle — which let us cheaply mint data in many custom
chain-of-thought (CoT) formats and keep only the derivable ones. (A known rule is not
the same as a rule *uniquely identifiable* from the shown examples — see the BIT section.
The solvers/validators were internal; the six generators written below are the *forward
samplers* only.) The piece we're happiest with: a clean way to **show** and **generate**
the hardest type, BIT, as `output = GATE(f(x), g(x), h(x))`, with an interactive viewer
to help others *visualize* the problem and watch the search space collapse.

- 🔗 **Repo (generators + interactive viewers):** https://github.com/definitelynotrussellkirk-bit/nemotron-puzzle-gen
- 🌐 **Live interactive pages:** https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/

> **Tone & scope.** Nothing here is a confident claim or a leaderboard result — it's
> a log of what we tried, what we observed, and what we now speculate. We hoped to
> transfer a search ability into the 30B; BIT results varied a lot (better runs ~50–60%,
> worst ~12% — a deterministic search where every step was locally solvable but the model
> couldn't chain them). Whether that was a failure on our part or something more
> fundamental, we honestly don't know. We've included lots of training traces for others.
> The generators below run live and contain **no raw competition rows**; a few sampling
> priors (e.g. the BIT gate-family weights) were estimated from *aggregate statistics* over
> competition data. More in *Budget & hindsight*.


## Why a generator platform

Four properties made the generators a fast CoT-experimentation loop:

1. **A free oracle.** Because we sampled the latent rule, an internal solver could
   verify *any* intermediate state for free, so we kept only derivable traces. (Those
   solvers were internal and are not part of this public release — the cells below
   write the forward samplers only.)
2. **Derivability as a hard gate.** A trace is allowed only if each step follows
   mechanically from the previous one (no naming a rule without the evidence, no
   decimal division a small model can't carry). A lint enforces it.
3. **A representation that's easy to show.** Reading BIT as one gate over three
   whole-byte streams, `GATE(f(x), g(x), h(x))`, instead of eight independent
   per-bit problems, is the cleanest thing to look at — and to generate.
4. **The data is over-determined — but *softly*.** On the whole-byte rule space (89,086
   functions, which covers 100% of real BIT puzzles), the exact *rule* is pinned only
   ~58% of the time and the *answer* is logically forced ~83%; yet picking the SIMPLEST
   consistent rule (Occam on arity) gets the exact answer ~99%. The answer is recovered by
   a simplicity prior, not by logic. See the BIT section below and `analysis/occam_solver.py`.

Because of (1)+(2), a $150 budget bought a lot of experimentation: no human labels,
no learned verifier to pay for. (Budget breakdown at the end.)


## The generators

Recreate the six samplers, then run them.

In [ ]:
import os
os.makedirs('generators', exist_ok=True)

In [ ]:
%%writefile generators/bit.py
#!/usr/bin/env python3
"""
BIT puzzle generator  —  output = GATE(f(x), g(x), h(x))
=========================================================

WHAT THE PUZZLE IS
------------------
Several (input_byte -> output_byte) examples of an unknown 8-bit boolean
transform, plus one query input.

HOW WE CREATE ONE (the generative model)
----------------------------------------
    1. pick a boolean GATE family       (e.g. OR_XNOR)
    2. pick a STREAM TRIPLE (f, g, h)    — three whole-byte views of the input
    3. for N random input bytes x, emit  x -> GATE(f(x), g(x), h(x))
    4. hold one pair out as the QUERY.

A "stream" is the input run through one cheap whole-byte op (shift / rotate /
complement / identity). The output byte is one gate applied across three streams.

ALIAS NOTE (matters when picking streams)
-----------------------------------------
For an 8-bit byte, rol_k == ror_(8-k). So the 58 *named* streams below are only
44 *distinct functions* — two names can denote the same op.
"""

import argparse
import json
import random

MASK = 0xFF


# ----------------------------------------------------------------------
# Whole-byte stream operations
# ----------------------------------------------------------------------
def rol(x, k):
    """Rotate-left by k bits (wrap-around)."""
    return ((x << k) | (x >> (8 - k))) & MASK


def ror(x, k):
    """Rotate-right by k bits (wrap-around)."""
    return ((x >> k) | (x << (8 - k))) & MASK


def stream_catalog():
    """
    {name: fn(x) -> byte} for every stream.

        x, ~x                       identity + complement      (2)
        shl1..7, shr1..7            logical shift, zero-fill   (14)
        rol1..7, ror1..7            rotate, wrap-around        (14)
        ~shl/~shr/~rol/~ror         complement of each above   (28)
    """
    s = {"x": lambda x: x & MASK, "~x": lambda x: (~x) & MASK}
    for k in range(1, 8):
        s[f"shl{k}"] = lambda x, k=k: (x << k) & MASK
        s[f"shr{k}"] = lambda x, k=k: (x >> k) & MASK
        s[f"rol{k}"] = lambda x, k=k: rol(x, k)
        s[f"ror{k}"] = lambda x, k=k: ror(x, k)
        s[f"~shl{k}"] = lambda x, k=k: (~(x << k)) & MASK
        s[f"~shr{k}"] = lambda x, k=k: (~(x >> k)) & MASK
        s[f"~rol{k}"] = lambda x, k=k: (~rol(x, k)) & MASK
        s[f"~ror{k}"] = lambda x, k=k: (~ror(x, k)) & MASK
    return s


STREAMS = stream_catalog()


# ----------------------------------------------------------------------
# Gate families  (h is the OR / selector term where relevant; ignored for
# 2-input families, kept in a uniform 3-arg signature for simplicity)
# ----------------------------------------------------------------------
GATES = {
    "OR_XNOR":          lambda f, g, h: (h | (f & g) | ((~f) & (~g))) & MASK,
    "GATED_XNOR_NAND":  lambda f, g, h: (((~h) & ((f & g) | ((~f) & (~g)))) | (h & (~(f & g)))) & MASK,
    "CH":               lambda f, g, h: ((f & g) | ((~f) & h)) & MASK,
    "MAJ3":             lambda f, g, h: ((f & g) | (f & h) | (g & h)) & MASK,
    "PAR3":             lambda f, g, h: (f ^ g ^ h) & MASK,
    "AO":               lambda f, g, h: ((f & g) | h) & MASK,
    "AX":               lambda f, g, h: ((f & g) ^ h) & MASK,
    "AND":              lambda f, g, h: (f & g) & MASK,          # h ignored
    "OR":               lambda f, g, h: (f | g) & MASK,          # h ignored
    "XOR":              lambda f, g, h: (f ^ g) & MASK,          # h ignored
    "AND_NOT":          lambda f, g, h: (g & (~f)) & MASK,       # h ignored
}
_TWO_INPUT = {"AND", "OR", "XOR", "AND_NOT"}

# Empirical family frequency on the 1,602 competition rows. Sampling with these
# weights yields a realistic family mix (uniform-over-families would massively
# over-represent the exotic 3-input gates, which are ~1% of real bits).
GATE_WEIGHTS = {
    "OR_XNOR": 72, "GATED_XNOR_NAND": 40, "OR": 30, "AND": 30, "XOR": 25,
    "CH": 5, "AND_NOT": 3, "MAJ3": 2, "AO": 2, "AX": 2, "PAR3": 1,
}


def _fmt(b):
    return format(b & MASK, "08b")


def sample(rng, n_examples=8):
    """
    Create ONE bit puzzle.

    Returns a dict:
        examples : list of {"input","output"} (8-char binary strings)
        query    : 8-char binary string
        answer   : 8-char binary string  (the gate applied to the query)
        rule     : {"gate","f","g","h"}  (exactly how it was built)
    """
    gate = rng.choices(list(GATE_WEIGHTS), weights=list(GATE_WEIGHTS.values()))[0]
    fn = GATES[gate]
    two = gate in _TWO_INPUT

    names = list(STREAMS)
    f = rng.choice(names)
    g = rng.choice(names)
    h = rng.choice(names) if not two else f

    def apply(x):
        return fn(STREAMS[f](x), STREAMS[g](x), STREAMS[h](x))

    xs = rng.sample(range(256), n_examples + 1)   # distinct inputs + query
    examples = [{"input": _fmt(x), "output": _fmt(apply(x))} for x in xs[:-1]]
    q = xs[-1]
    return {
        "type": "bit",
        "examples": examples,
        "query": _fmt(q),
        "answer": _fmt(apply(q)),
        "rule": {"gate": gate, "f": f, "g": g, "h": (None if two else h)},
    }


def main():
    ap = argparse.ArgumentParser(description="Create BIT puzzles: GATE(f,g,h).")
    ap.add_argument("-n", "--num", type=int, default=3, help="how many to create")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--examples", type=int, default=8, help="examples per puzzle")
    ap.add_argument("--jsonl", action="store_true", help="emit JSONL instead of pretty")
    args = ap.parse_args()
    rng = random.Random(args.seed)

    for _ in range(args.num):
        p = sample(rng, args.examples)
        if args.jsonl:
            print(json.dumps(p))
            continue
        r = p["rule"]
        print(f"\nRULE  {r['gate']}(f={r['f']}, g={r['g']}, h={r['h']})")
        for e in p["examples"]:
            print(f"  {e['input']} -> {e['output']}")
        print(f"  QUERY {p['query']} -> {p['answer']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile generators/encryption.py
#!/usr/bin/env python3
"""
ENCRYPTION puzzle generator  —  1:1 letter substitution over a fixed vocabulary
===============================================================================

WHAT THE PUZZLE IS
------------------
A monoalphabetic substitution cipher: every plaintext letter maps to a unique
cipher letter (a bijection on a..z), spaces preserved. All plaintext words come
from a small FIXED VOCABULARY. You see a few (cipher -> plain) sentences, then
decode a query.

HOW WE CREATE ONE (the generative model)
----------------------------------------
    1. fix a vocabulary V (the competition used a 77-word list; here a compact
       illustrative set).
    2. sample a random bijection  key : a..z -> a..z   (the cipher).
    3. build example sentences and a query from V, encode them with `key`.

The closed vocabulary is what keeps instances tractable: every plaintext word is
guaranteed to be in V, so unknown cipher-words can be resolved by testing every
same-length vocabulary word. (Decoding is usually but not always unique — a short
query can occasionally admit more than one vocabulary-consistent reading.)
"""

import argparse
import json
import random
import string

# Compact illustrative vocabulary (closed world).
VOCAB = [
    "cat", "key", "map", "the", "and", "you",
    "bird", "book", "cave", "dark", "door", "king", "near", "wise",
    "queen", "reads", "river", "sees",
    "dragon", "castle", "golden", "secret",
    "princess", "treasure", "mountain", "explores",
]


def _random_key(rng):
    """A random bijection a..z -> a..z (the substitution cipher)."""
    letters = list(string.ascii_lowercase)
    shuffled = letters[:]
    rng.shuffle(shuffled)
    return dict(zip(letters, shuffled))


def _encode(word, key):
    return "".join(key[c] for c in word)


def sample(rng, n_examples=4):
    """
    Create ONE encryption puzzle.

    Returns:
        examples : list of {"cipher","plain"} sentences (space-separated)
        query    : cipher sentence to decode
        answer   : plaintext sentence
        rule     : {"key": {plain_letter: cipher_letter, ...}}  (the cipher used)
    """
    key = _random_key(rng)
    examples = []
    for _ in range(n_examples):
        words = [rng.choice(VOCAB) for _ in range(rng.randint(3, 5))]
        examples.append({
            "plain": " ".join(words),
            "cipher": " ".join(_encode(w, key) for w in words),
        })
    qwords = [rng.choice(VOCAB) for _ in range(rng.randint(2, 4))]
    return {
        "type": "encryption",
        "examples": examples,
        "query": " ".join(_encode(w, key) for w in qwords),
        "answer": " ".join(qwords),
        "rule": {"key": key},
    }


def main():
    ap = argparse.ArgumentParser(description="Create ENCRYPTION (substitution) puzzles.")
    ap.add_argument("-n", "--num", type=int, default=3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--examples", type=int, default=4)
    ap.add_argument("--jsonl", action="store_true")
    args = ap.parse_args()
    rng = random.Random(args.seed)

    for _ in range(args.num):
        p = sample(rng, args.examples)
        if args.jsonl:
            print(json.dumps(p))
            continue
        print("\nEXAMPLES")
        for e in p["examples"]:
            print(f"  {e['cipher']:<28} -> {e['plain']}")
        print(f"  QUERY {p['query']:<22} -> {p['answer']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile generators/transformation.py
#!/usr/bin/env python3
"""
TRANSFORMATION puzzle generator  —  two numbers -> hidden arithmetic recipe
===========================================================================

WHAT THE PUZZLE IS
------------------
Each example is "A op B = RESULT" where RESULT is produced by a hidden recipe
with THREE independent knobs:
    1. OPERATION   : add, sub, mul, absdiff, mod, floordiv, concat, ...
    2. ORDERING    : operands taken as (A,B) or swapped (B,A), optionally with
                     each operand's digits reversed first.
    3. STYLE       : how the result is written -- plain, digit-reversed,
                     sign-prefixed ("-12"), or sign-suffixed ("12-").

HOW WE CREATE ONE (the generative model)
----------------------------------------
Pick (operation, ordering, style), sample operand pairs, render each example
through the recipe, hold one out as the query. The three knobs are independent,
so the result string carries an encoding (ordering + style) on top of the raw
arithmetic -- that layering is the whole point of the type.

CIPHER VARIANT
--------------
`sample(..., cipher=True)` disguises digits 0-9 as symbols via a random bijection;
the symbol map is implied by the examples.
"""

import argparse
import json
import random

OPS = {
    "add":      lambda a, b: a + b,
    "sub":      lambda a, b: a - b,
    "mul":      lambda a, b: a * b,
    "absdiff":  lambda a, b: abs(a - b),
    "mod":      lambda a, b: a % b if b else 0,
    "floordiv": lambda a, b: a // b if b else 0,
    "concat":   lambda a, b: int(f"{a}{b}"),
}
OP_WEIGHTS = {"add": 5, "sub": 4, "mul": 4, "absdiff": 3, "mod": 2, "floordiv": 2, "concat": 2}

ORDERINGS = ["AB", "BA", "AB_rev", "BA_rev"]   # _rev = reverse each operand's digits first
STYLES = ["plain", "rev", "opsign", "tailsign"]


def _rev(n):
    return int(str(n)[::-1] or "0")


def _operands(a, b, ordering):
    if ordering == "AB":
        return a, b
    if ordering == "BA":
        return b, a
    if ordering == "AB_rev":
        return _rev(a), _rev(b)
    if ordering == "BA_rev":
        return _rev(b), _rev(a)
    raise ValueError(ordering)


def _style(value, style):
    """Render a numeric result string under the chosen style."""
    neg = value < 0
    digits = str(abs(value))
    if style == "plain":
        return ("-" if neg else "") + digits
    if style == "rev":
        return ("-" if neg else "") + digits[::-1]
    if style == "opsign":          # always prefix a marker
        return "-" + digits
    if style == "tailsign":        # marker as suffix
        return digits + "-"
    raise ValueError(style)


def render(a, b, op, ordering, style):
    x, y = _operands(a, b, ordering)
    return _style(OPS[op](x, y), style)


def _digit_map(rng):
    syms = list("@#$%&*+=?!")
    rng.shuffle(syms)
    return {str(d): syms[d] for d in range(10)}


def _enc(s, dmap):
    """Encode a numeric string (may carry a sign char) symbol-by-symbol."""
    return "".join(dmap.get(c, c) for c in str(s))


def sample(rng, n_examples=4, cipher=False):
    """
    Create ONE transformation puzzle.

    Returns:
        examples : list of {"a","b","result"} (symbol-encoded if cipher)
        query    : {"a","b"}
        answer   : result string
        rule     : {"op","ordering","style"[,"digit_map"]}  (how it was built)
    """
    op = rng.choices(list(OP_WEIGHTS), weights=list(OP_WEIGHTS.values()))[0]
    ordering = rng.choice(ORDERINGS)
    style = rng.choice(STYLES)
    dmap = _digit_map(rng) if cipher else None

    def enc(s):
        return _enc(s, dmap) if cipher else str(s)

    pairs = [(rng.randint(10, 99), rng.randint(10, 99)) for _ in range(n_examples + 1)]
    examples = [{"a": enc(a), "b": enc(b), "result": enc(render(a, b, op, ordering, style))}
                for a, b in pairs[:-1]]
    qa, qb = pairs[-1]
    rule = {"op": op, "ordering": ordering, "style": style}
    if cipher:
        rule["digit_map"] = dmap
    return {
        "type": "transformation_cipher" if cipher else "transformation",
        "examples": examples,
        "query": {"a": enc(qa), "b": enc(qb)},
        "answer": enc(render(qa, qb, op, ordering, style)),
        "rule": rule,
    }


def main():
    ap = argparse.ArgumentParser(description="Create TRANSFORMATION puzzles (numeric / cipher).")
    ap.add_argument("-n", "--num", type=int, default=3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--examples", type=int, default=4)
    ap.add_argument("--cipher", action="store_true", help="symbol-disguised digits")
    ap.add_argument("--jsonl", action="store_true")
    args = ap.parse_args()
    rng = random.Random(args.seed)

    for _ in range(args.num):
        p = sample(rng, args.examples, cipher=args.cipher)
        if args.jsonl:
            print(json.dumps(p))
            continue
        r = p["rule"]
        print(f"\nRULE  op={r['op']} ordering={r['ordering']} style={r['style']}")
        for e in p["examples"]:
            print(f"  {e['a']} ? {e['b']} = {e['result']}")
        print(f"  QUERY {p['query']['a']} ? {p['query']['b']} = {p['answer']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile generators/gravitational.py
#!/usr/bin/env python3
"""
GRAVITATIONAL puzzle generator  —  d = 1/2 * g * t^2
====================================================

WHAT THE PUZZLE IS
------------------
A falling-distance relation with an UNKNOWN gravitational constant g. A few
(t, d) examples, then predict d for a query t.

HOW WE CREATE ONE (the generative model)
----------------------------------------
    1. pick an unknown g.
    2. for several times t, emit d = 0.5 * g * t^2  (rounded for display).
    3. query a fresh t.

The constant rate across examples is r = d / t^2 = g/2. Displayed distances are
rounded, so instances are graded with a small relative tolerance.
"""

import argparse
import json
import random


def sample(rng, n_examples=3, round_dp=2):
    """
    Create ONE gravitational puzzle.

    Returns:
        examples : list of {"t","d"}  (d rounded to round_dp)
        query    : {"t"}
        answer   : d for the query (rounded)
        rule     : {"g": value, "rate": g/2}  (how it was built)
        tolerance: suggested relative tolerance for grading
    """
    g = round(rng.uniform(1.5, 25.0), 2)
    rate = g / 2.0
    ts = rng.sample([round(t, 1) for t in (v / 2 for v in range(2, 25))], n_examples + 1)

    def d(t):
        return round(rate * t * t, round_dp)

    return {
        "type": "gravitational",
        "examples": [{"t": t, "d": d(t)} for t in ts[:-1]],
        "query": {"t": ts[-1]},
        "answer": d(ts[-1]),
        "rule": {"g": g, "rate": rate},
        "tolerance": 0.01,
    }


def main():
    ap = argparse.ArgumentParser(description="Create GRAVITATIONAL puzzles (d=1/2 g t^2).")
    ap.add_argument("-n", "--num", type=int, default=3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--examples", type=int, default=3)
    ap.add_argument("--jsonl", action="store_true")
    args = ap.parse_args()
    rng = random.Random(args.seed)

    for _ in range(args.num):
        p = sample(rng, args.examples)
        if args.jsonl:
            print(json.dumps(p))
            continue
        print(f"\nRULE  g={p['rule']['g']}  (rate=d/t^2={p['rule']['rate']})")
        for e in p["examples"]:
            print(f"  t={e['t']:<5} d={e['d']}")
        print(f"  QUERY t={p['query']['t']} -> d={p['answer']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile generators/unit_conversion.py
#!/usr/bin/env python3
"""
UNIT CONVERSION puzzle generator  —  out = in * factor
======================================================

WHAT THE PUZZLE IS
------------------
A linear conversion with an UNKNOWN factor. A few (input, output) pairs, then
convert a query input.

HOW WE CREATE ONE (the generative model)
----------------------------------------
    1. pick an unknown positive factor.
    2. for several inputs, emit  out = in * factor  (rounded for display).
    3. query a fresh input.

The constant rate across examples is f = out / in. Displayed outputs are rounded,
so instances are graded with a small relative tolerance.
"""

import argparse
import json
import random


def sample(rng, n_examples=3, round_dp=2):
    """
    Create ONE unit-conversion puzzle.

    Returns:
        examples : list of {"in","out"}  (out rounded to round_dp)
        query    : {"in"}
        answer   : converted output (rounded)
        rule     : {"factor": value}  (how it was built)
        tolerance: suggested relative tolerance for grading
    """
    factor = round(rng.uniform(0.05, 40.0), 3)
    ins = rng.sample([round(v, 1) for v in (n / 2 for n in range(2, 60))], n_examples + 1)

    def out(v):
        return round(v * factor, round_dp)

    return {
        "type": "unit_conversion",
        "examples": [{"in": v, "out": out(v)} for v in ins[:-1]],
        "query": {"in": ins[-1]},
        "answer": out(ins[-1]),
        "rule": {"factor": factor},
        "tolerance": 0.01,
    }


def main():
    ap = argparse.ArgumentParser(description="Create UNIT CONVERSION puzzles (out=in*factor).")
    ap.add_argument("-n", "--num", type=int, default=3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--examples", type=int, default=3)
    ap.add_argument("--jsonl", action="store_true")
    args = ap.parse_args()
    rng = random.Random(args.seed)

    for _ in range(args.num):
        p = sample(rng, args.examples)
        if args.jsonl:
            print(json.dumps(p))
            continue
        print(f"\nRULE  factor={p['rule']['factor']}")
        for e in p["examples"]:
            print(f"  in={e['in']:<6} out={e['out']}")
        print(f"  QUERY in={p['query']['in']} -> out={p['answer']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile generators/number_conversion.py
#!/usr/bin/env python3
"""
NUMBER CONVERSION puzzle generator  —  integer <-> Roman numeral
================================================================

WHAT THE PUZZLE IS
------------------
Convert between Arabic integers and Roman numerals. The examples establish the
direction (int->roman or roman->int); the query asks for one more conversion.

HOW WE CREATE ONE (the generative model)
----------------------------------------
    1. pick a direction.
    2. sample integers in [1, 3999], render each side with the fixed value table.
    3. query a fresh integer.

Roman numerals are a fixed, table-driven additive/subtractive notation (not a
positional system), so rendering is a deterministic greedy walk of the value
table -- no unknown parameter at all.
"""

import argparse
import json
import random

_VALUES = [
    (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
    (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
    (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
]


def to_roman(n):
    """Greedy value-table decode: append each numeral while it fits."""
    out = []
    for v, sym in _VALUES:
        while n >= v:
            out.append(sym)
            n -= v
    return "".join(out)


def sample(rng, n_examples=3, direction=None):
    """
    Create ONE number-conversion puzzle.

    Returns:
        direction : "int2roman" or "roman2int"
        examples  : list of {"input","output"}
        query     : {"input"}
        answer    : converted value
        rule      : {"direction": ...}  (how it was built)
    """
    direction = direction or rng.choice(["int2roman", "roman2int"])
    nums = rng.sample(range(1, 4000), n_examples + 1)

    def render(n):
        if direction == "int2roman":
            return {"input": str(n), "output": to_roman(n)}
        return {"input": to_roman(n), "output": str(n)}

    examples = [render(n) for n in nums[:-1]]
    q = render(nums[-1])
    return {
        "type": "number_conversion",
        "direction": direction,
        "examples": examples,
        "query": {"input": q["input"]},
        "answer": q["output"],
        "rule": {"direction": direction},
    }


def main():
    ap = argparse.ArgumentParser(description="Create NUMBER CONVERSION puzzles (int<->roman).")
    ap.add_argument("-n", "--num", type=int, default=3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--examples", type=int, default=3)
    ap.add_argument("--direction", choices=["int2roman", "roman2int"], default=None)
    ap.add_argument("--jsonl", action="store_true")
    args = ap.parse_args()
    rng = random.Random(args.seed)

    for _ in range(args.num):
        p = sample(rng, args.examples, args.direction)
        if args.jsonl:
            print(json.dumps(p))
            continue
        print(f"\nRULE  direction={p['direction']}")
        for e in p["examples"]:
            print(f"  {e['input']:<6} -> {e['output']}")
        print(f"  QUERY {p['query']['input']} -> {p['answer']}")


if __name__ == "__main__":
    main()


### Live: create puzzles

Each call picks a hidden rule, samples inputs, and emits `examples + query + answer + the exact rule used`. The outputs below are real, captured when this notebook was built.

In [ ]:
import random, sys
sys.path.insert(0, 'generators')
import bit
rng = random.Random(0)
p = bit.sample(rng)
print('BIT  rule =', p['rule'])
for e in p['examples'][:4]:
    print(' ', e['input'], '->', e['output'])
print('  QUERY', p['query'], '->', p['answer'])

BIT  rule = {'gate': 'XOR', 'f': '~rol6', 'g': '~rol7', 'h': None}
  11010111 -> 00011110
  00010100 -> 00001111
  10000100 -> 01100011
  11111000 -> 01000010
  QUERY 01101111 -> 01101100


In [ ]:
import random, sys
sys.path.insert(0, 'generators')
import encryption, transformation, gravitational, unit_conversion, number_conversion
rng = random.Random(1)
for mod in (encryption, transformation, gravitational, unit_conversion, number_conversion):
    q = mod.sample(rng)
    print(f"\n{q['type']}  rule={q['rule']}")
    print('  example:', q['examples'][0])
    print('  query  :', q['query'], '->', q['answer'])


encryption  rule={'key': {'a': 'x', 'b': 'y', 'c': 'l', 'd': 'k', 'e': 'w', 'f': 'b', 'g': 'f', 'h': 'z', 'i': 't', 'j': 'n', 'k': 'j', 'l': 'r', 'm': 'q', 'n': 'a', 'o': 'h', 'p': 'v', 'q': 'g', 'r': 'm', 's': 'u', 't': 'o', 'u': 'p', 'v': 'd', 'w': 'i', 'x': 'c', 'y': 's', 'z': 'e'}}
  example: {'plain': 'cat golden sees', 'cipher': 'lxo fhrkwa uwwu'}
  query  : uwlmwo yhhj -> secret book

transformation  rule={'op': 'mod', 'ordering': 'AB_rev', 'style': 'plain'}
  example: {'a': '63', 'b': '81', 'result': '0'}
  query  : {'a': '52', 'b': '74'} -> 25

gravitational  rule={'g': 23.51, 'rate': 11.755}
  example: {'t': 7.5, 'd': 661.22}
  query  : {'t': 4.0} -> 188.08

unit_conversion  rule={'factor': 12.17}
  example: {'in': 19.5, 'out': 237.31}
  query  : {'in': 28.0} -> 340.76

number_conversion  rule={'direction': 'roman2int'}
  example: {'input': 'MMCDXIII', 'output': '2413'}
  query  : {'input': 'MCMLXVIII'} -> 1968


## Interactive viewers (best viewed live)

Three browser tools ship with the repo. Kaggle's *published* view strips interactive
JS, so they are best opened live on the project site:

- **BIT viewer** — pick three byte-streams and a gate, watch the output byte computed
  bit-by-bit, with the real per-bit data distribution → https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/
- **Over-determination explorer** — sample K examples from a rule and watch the
  candidate hypotheses shrink one example at a time, with a uniqueness verdict
  (exact within its catalog) → https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/over-determination.html
- **Mini-skill gallery** — shuffle a small random sample of the atomic drills, filter
  by type, pull another generation of any skill → https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/miniskills.html

We hope the BIT viewer helps others *visualize* the bit problems and how the search
space collapses — using an LLM to build quick data visualizations, which many across
past competitions have found valuable.

*To embed one live in a fork:* `from IPython.display import IFrame` then
`IFrame("https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/", width="100%", height=720)`.


## Per-type insights

**BIT — `GATE(f,g,h)`.** A stream is the input through one cheap whole-byte op
(`shl/shr/rol/ror`, complement, identity). Output = one gate across three streams;
dominant families are `OR_XNOR` and a `XNOR/NAND` selector. For an 8-bit byte
`rol_k == ror_(8-k)`, so 58 named streams are only **44 distinct functions** —
score the function, not the label.

**BIT: a search for FUNCTIONS vs a search for ANSWERS.** The whole-byte `GATE(f,g,h)`
class has **89,086** distinct functions, and it reproduces **100%** of the real BIT
puzzles (every gold answer is a gate over streams).
Measured on the real train.csv puzzles (median 9 examples), success depends entirely on
*which question you ask* (reproducible: `analysis/occam_solver.py`):

| Question | How often it works |
|---|---|
| pin the exact RULE (function search) | ~58% |
| all consistent rules agree on the ANSWER | ~83% |
| pick a *random* consistent rule | ~93% |
| **pick the SIMPLEST consistent rule (Occam on arity)** | **~99%** |
| ignore the examples (random byte) | 0.39% |

So the answer is **not logically forced** — it's recovered by a **prior**: the generator
picks simple rules, so the simplest consistent rule is almost always the true one, and
Occam-on-arity lands the exact answer ~99% on random examples (our internal solver scored
~99.3% on the real competition rows, measured on our machine; train.csv isn't shipped here,
so that figure isn't reproducible from this notebook). That's
*soft*, prior-recovered over-determination, not hard logic. In hindsight our effort went
into the harder framing (**function search**, underdetermined ~42% of the time); the easy
target was **answer search** — and Tong's per-column / frequency-prior method reads like
exactly that. Our `occam_solver.py` "Autotracer" is the answer-search form: paste examples
+ a query, it derives the simplest consistent rule and the CoT.

**Encryption — closed vocabulary.** Every plaintext word is in a fixed list, so
unknown cipher-words are finished by exhaustive same-length candidate testing +
bijection rejection. That closure is the whole reason the type is exact.

**Transformation — the one piece of ours that actually shipped.** operation ×
ordering × style, layered as an encoding on the raw arithmetic. We left the *numeric*
lane unexplored, but the **cipher (cryptarithm) transform rows** from this generator —
about **2,500 verified rows** (a CSP solve for the symbol→digit map + honest priors) —
were the one part of our own work that made it into our best scoring blend, on top of
a strong public CoT base. Honest caveat: we couldn't cleanly attribute the lift, and
cipher-type accuracy stayed ~7.5%. The numeric lane is the most open next step.

**Numeric rates — clear the decimals first, like a human.** For `d = ½·g·t²` and
`out = in·factor`, cancel the hidden decimal constant by **ratio** instead of
recovering it: `d_q = d_e·(t_q/t_e)²`. The unknown divides out and the trace
collapses to one step. Honest caveat: the shown values are *rounded*, so this is a
**tolerance-compatible shortcut**, not exact cancellation — in a quick simulation it
hit the exact 2-decimal answer ~74% (gravitational) / ~68% (unit) from one example,
with nearly all predictions inside the task's 1% tolerance.

**Number conversion — no hidden parameter.** int↔Roman is a fixed, table-driven
additive/subtractive notation (not positional) — a deterministic greedy walk of the
value table; the examples only signal direction.


## The CoT formats were search traces — approaches & papers

Most custom formats are **search traces** (enumerate candidate rules → test → prune
→ lock). Our initial framing was **recursive bisection**: to get from a state `A`
to the goal `Z`, if it fails in one shot, introduce a waypoint `N`, solve `A→N` and
`N→Z` separately, and keep bisecting until each hop is trivial. That divide-and-
conquer instinct led us to the search-supervision literature, whose training shapes
are the same idea made trainable. We *hoped* one thing would be our edge: our solver
is an exact, free verifier of every intermediate state, so the weak/learned-verifier
bottleneck the literature describes wouldn't apply to us. It didn't translate into a
solving win for us (see the dissociation note and budget below).

- **Teach-long enumerate→test→prune SFT** — every step a solver-checkable
  state→action→observation→prune transition; rejected branches shown in-trace.
- **STaR / expert-iteration trim** — *inspired by* Zelikman et al., *STaR* (2022)
  and Lehnert et al., *Searchformer / "Beyond A\*"* (2024). Those papers filter
  sampled rationales / learn A\* dynamics and bootstrap shorter traces; "keep the
  shortest **verified** trace under a declining length budget, else the canonical
  trace" is *our* adaptation, not their method.
- **GRPO process / progress reward** — GRPO is from Shao et al., *DeepSeekMath*
  (2024, arXiv:2402.03300); the Hamming-distance / "gold still reachable" rewards
  are *our* proposed shaping, not from that paper.
- **Step-level process supervision** — Lightman et al., *Let's Verify Step by Step*
  (2023). Our solver emits the earliest-wrong-step label for free.
- **Off the table at submission:** tree-of-thought / MCTS / self-consistency —
  competition inference is greedy single-pass, so these stayed training-data tools.

**Compared to Tong Huikang's 0.85 (speculation).** We used *first divergence* — the
first example/position where candidate rules disagree — to build our atomic A→B
drills, much like Tong's per-column method (freq priors + stride extrapolation;
[github.com/tonghuikang/nemotron](https://github.com/tonghuikang/nemotron)).
Supportable facts: his repo presents a Progress-Prize submission and the linked
notebook is titled "End-to-end finetuning for LB 0.85". *Speculating* from the public
repo (we did not reverse-engineer it): we wonder whether his traces baked the bit
computation, the search priors, *and* the answer into **one coherent trace** — and
that "all in one trace" was the important move. That's a testable hypothesis, not a
verified comparison. We instead fragmented the same
material across a library of micro-skills and many CoT formats; the pieces were
there, but the model may never have seen one trace teaching search + the bit op +
the answer together. Consolidating into a single unified trace is near the top of
our "if we ran it again" list.


## A divergence we *think* we saw: rank *given* candidates vs *generate* them

A BIT answer is one byte — only 256 possibilities — so we tried turning solving into
**ranking**: hand the model a shortlist (64 → 32 → 16 → 8 candidates) and have it
pick the most likely. Given the shortlist, it *seemed* to pick reliably. When it had to
**produce** the shortlist itself — even from a deterministic recipe — accuracy seemed to
fall to ~78–80%. (Earlier slate formats had parsing failures; the K=8 format we report
parsed 64/64, so the loss looked like candidate *recall*, not formatting.)

On a small (64-row) eval we tried to factorize that condition
(end-to-end ≈ parse × candidate-set × selection-given-set):

| Stage (generate-then-pick) | What we saw |
|---|---|
| **Parse** — clean, parseable 8-candidate set | 64/64 (always exactly 8) |
| **Candidate set correct** — its 8 contained/identified gold | ~78–80% (50–51/64) |
| **Selection given set correct** — picked gold given its set was right | 50/50, 51/51 |

So the errors we saw *seemed* to land in **candidate-set construction**: parsing held, and
on the 50–51 rows whose set contained gold we happened to see no selection errors. **Big
caveat:** 50–51 rows is a small sample with real variance — "no errors" is suggestive, not
proof that selection is perfect. The fuller picture:

| What the model had to do | Accuracy we saw (small evals) |
|---|---|
| Execute a *given* function (scaffolded) | up to ~100% (64/64, 512/512) |
| Generate 8 candidates, then pick | ~78–80% (50–51/64) |
| Solve raw end-to-end (no scaffold) | ~11–22% (7/64–14/64) |

As scaffolding was removed, accuracy went down. This is the same local-vs-global pattern
as the alphabet point in the budget section: the model seemed able to do each step alone
but not chain them.

For reference, "random" depends on what you average over (see the BIT section): ignoring
the examples is 0.39% (1/256); a guess that stays *consistent* with the examples is right
about 93%; the simplest consistent rule (Occam) about 99%. So the ~78–80% on the
generate-then-pick condition is below a random-consistent guess. Within-slate baselines:
12.5% (uniform over 8), 3.1% (a random 8-slate contains gold).
*(Source: `bit_alice_cache_slate_mask_qset_k8` / `qset_only_k8`, 64 rows, greedy decode;
LoRA-SFT BIT search lineage on Nemotron-3-Nano-30B — internal evals, not a leaderboard
number.)*


## Mini-skills — from holistic inspection of mistakes

Beneath the full traces sits a library of **239 atomic "mini-skill" drills**, each
designed from **holistic inspection of the model's mistakes**: we read *how* it
failed and distilled the missing micro-competence into a small, self-contained
drill (a single gate evaluation, a shift, a bijection check, a popcount-delta
sign, …). The base generators mint these on demand, so the library grew out of the
failure analysis rather than a fixed syllabus.

- **Browse:** `miniskills.html` — gallery covers **233 of 239 registered** skills
  (up to 2 random generations each)
- **Full random dump:** `miniskills_sample.jsonl` — 5,891 weighted-random rows,
  **221** of 239 skills (a random sample, so not all 239 appear)
- Live gallery: https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/miniskills.html

**The recipe (so you can build your own).** The model could often do each step of a
procedure but not chain them. So we generated one drill per step. It's task-decomposition
turned into data — not magic:

```
full task:   A ───────────────▶ Z
                  decompose into the steps a solver does
             A ─▶ B ─▶ C ─▶ … ─▶ Z

per step:  a sampler emits random instances of "given A, produce B"  (one generator each)
           plus one "chain A ─▶ … ─▶ Z" drill
gate each row on the solver  ─▶  keep only derivable ones  =  the dataset
```

List your solver's mechanical steps, write a sampler per step (and one for the whole
chain), and check each row against the solver. That transfers to any decomposable task.


## Budget & hindsight

We earmarked about **$150** at the start — roughly **$100 for Tinker** (LoRA SFT /
RL) and **~$50 / ~600 credits for Colab Pro+**. We kept each **experiment** cheap —
only about **$2 of Tinker compute apiece** — by running lots of small steps. To be
precise about the bookkeeping: an "experiment" means one format/idea, which usually
bundled several short training *runs* plus evals; the ~$100 spread over a few dozen
experiments, which is why the per-experiment figure and the larger run count below
aren't in tension. Exact, free oracles meant every generated trace was usable
without human labels or a learned verifier; and **having CODEX (an agent) manage the
Tinker runs end-to-end — launch a small step, evaluate, hand off, repeat, no idle GPU
or manual babysitting — saved a lot of money** and made the volume affordable.

**Trace length was our other cost lever.** Shorter chains-of-thought are cheaper to
train on and to run at inference (output is capped at 7,800 tokens), so for most of the
competition we kept traces tight — aiming under **~900 tokens**, then relaxing to
**~1.1k**, then **~1.3k** as harder cases needed room. Only near the end did we spend the
*full* budget, letting the enumerate-test-prune search traces run long to try to teach
BIT — where cost climbed. On training, we had good luck with **QLoRA (4-bit LoRA) on an
H100** (via Colab Pro+, alongside Tinker for the LoRA SFT/RL), especially at the *lower*
token counts where short traces packed efficiently and runs stayed fast and cheap.

**The scale that bought:** roughly **950 Tinker runs** (≈500 training + ≈450 eval)
over **~33 days**, about **175** training-data builds, the BIT trace format revised
through **30-plus numbered versions** (up to V35), and **256** automated handoff
notes between agents. Small steps, kept cheap, run at volume.

**We had a strong early start — so we were hopeful, not going in blind.** Before we
leaned on a public CoT base, one of our own SFT-trace submissions already scored about
**78% overall** — which we *believe* placed us around **top 50** on the leaderboard at the
time — notably with BIT at only ~**22%**, so the five other puzzle types were already
carrying the score. That early traction kept us hopeful, and throughout we kept
doing **holistic looks at the errors** — reading what the model got wrong and distilling
each gap into a targeted mini-skill, rather than guessing. BIT was the obvious holdout,
which is what pushed us into the search-teaching experiments (which, on BIT specifically,
varied widely — from a worst of ~12% up to ~50–60% in better runs; same hard type).

**Where the time went, and what our final submission was.** Most of our effort went
into trying to teach the model a *different* search method for BIT (the bisection /
enumerate-test-prune work above) — and that bulk never became a finished submission.
Results **varied a lot**: our better BIT runs reached roughly **50–60%**, an early attempt
was ~**22%**, and the *worst* — a deterministic search that fit within budget — got only
~**12%**. In that ~12% case everything was **locally solvable** — like the alphabet: ask the
model each step alone ("what comes after A?", "after B?") and it's right essentially every
time; ask it to recite the whole alphabet and it drops to ~12%. It did each `A→B` step in
isolation but did not chain them — the same pattern as the dissociation above. (Our final
leaderboard entry was small and almost incidental: we were testing whether our cipher
rows added lift, and by the time we looked up we'd run out of time.)

**If we went again, we'd prototype on a small model first** — *for experiments, not for
submission.* The competition target is fixed at Nemotron-3-Nano-30B-A3B; we just mean a
cheap prototyping rig to iterate the search-teaching loop faster, then transfer the
lessons up to the fixed 30B (never submitting a different size). Iterating directly on a
30B is slow. Much of what *seems* to matter — trace shape, length budget, verifier-gated
trimming, where the model fakes a step — *looks* largely model-agnostic, so we suspect
those lessons would transfer up from a smaller model at a fraction of the cost. We can't
prove it; it's the bet we'd make next time. The unexplored transformation lane and
the STaR/GRPO loops are where we'd look first.

**SPECULATION (hindsight-inspired) — an architecture hunch about MoE routing.** *(All of
this is a hindsight-inspired hunch we could not confirm — us connecting our own routing
struggles, after the fact, with a noise result we only really weighed later.)*
Nemotron-3-Nano-30B-A3B is a mixture-of-experts: almost all of its capacity is in the
sparse *experts*, not the shared/attention layers. We reached about **0.76 relatively
easily with LoRA on the shared layers only — no expert layers**, which already gets the
bulk of the non-BIT types. Pushing further on BIT seemed to need the experts, and in
hindsight we *feel* the hard part was keeping expert **routing consistent across a long
trace**: a long enumerate-test-prune BIT trace only holds together if the same experts
fire reliably step after step, so any routing drift reads as noise. We *speculate* (purely
a guess) that some of that noise may even have been expert normalization. We're not the
only ones poking at noise here — we recall a public **gold-medal notebook** reporting that
simply *adding noise* improved their result, and *perhaps that's why*: if consistent
expert routing is the real bottleneck, a little noise (or a different normalization) might
help in a way none of us fully understands. We never pinned it down — flagging it in case
it's a thread someone can pull.

**Why share something that hasn't paid off?** It hasn't produced a reliable result
for us *so far* — BIT runs ranged from ~12% to ~50–60%. Whether that was a failure on our part or
something more fundamental, we honestly don't know. We suspect the missing piece may be
mundane (not enough training, too small a budget, or our cheap small-model *experiments*
not transferring up to the fixed 30B target — the submission model size is the
competition's, not ours to change). So we've
included lots of **training traces** (the 5,891-row mini-skill dump and the CoT
formats) to give others ideas to build on. And part of it is paying it forward: reading
other people's public notebooks gave *us* insights along the way (the noise observation
above is one), so contributing ours back is the least we can do. Putting it in the open
seems like the fastest way for someone to spot what we missed — that's the point of an
open contribution.


## Reproduce / explore

- All six generators and the three interactive viewers: **https://github.com/definitelynotrussellkirk-bit/nemotron-puzzle-gen**
- Live pages (guaranteed interactive): **https://definitelynotrussellkirk-bit.github.io/nemotron-puzzle-gen/**

```bash
python3 generators/bit.py -n 5 --seed 0            # create 5 BIT puzzles
python3 generators/bit.py -n 1000 --jsonl > bit.jsonl
python3 generators/transformation.py --cipher -n 3 # symbol-disguised digits
```

**Acknowledgments.** Thanks to **Tong Huikang**, whose public Progress-Prize 0.85
solution ([github.com/tonghuikang/nemotron](https://github.com/tonghuikang/nemotron))
was the reference we compared our BIT approach against and learned from. Programming and
task management throughout were done with the help of AI coding agents — **OpenAI's
Codex** and **Anthropic's Claude**; Codex in particular drove the long-running Tinker
experiment loops end-to-end.

*Category entered: Best Data / Synthetic Data Method.*
